# Week 15, Day 3: The Poisson distribution: limits, intervals and over-dispersion

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w015/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: How fast does the binomial become Poisson?

Write `poisson_pmf(k, lam)` and `binom_pmf(k, n, p)` with the `math` module. For λ = 3 and n = 10, 100 and 1000, compute the total variation distance, half the sum over k = 0..30 of |binomial − Poisson|, and store the three values in `tv`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, exp, factorial

def poisson_pmf(k, lam):
    return ...

def binom_pmf(k, n, p):
    return ... if k <= n else 0.0

lam = 3
tv = [0.5 * sum(abs(binom_pmf(k, n, lam / n) - poisson_pmf(k, lam)) for k in range(31)) for n in [10, 100, 1000]]
print("total variation distance for n = 10, 100, 1000:", np.round(tv, 5))

In [ ]:
import numpy as np
def test_pmfs():
    "both PMFs sum to almost 1 over 0..30"
    assert abs(sum(poisson_pmf(k, 3) for k in range(31)) - 1) < 1e-9
    assert abs(sum(binom_pmf(k, 100, 0.03) for k in range(31)) - 1) < 1e-9

def test_shrinks():
    "the total variation distance falls roughly like 1/n"
    assert tv[0] > tv[1] > tv[2] and 5 < tv[0] / tv[1] < 15 and 5 < tv[1] / tv[2] < 15

test_pmfs()
test_shrinks()
print("All checks passed")

### Exercise: Memoryless and not

Draw 200,000 intervals at 20 Hz, once exponential and once gamma with shape 4 and the same mean. For each, compute the unconditional P(wait > 50 ms) and the conditional P(wait > 100 ms | wait > 50 ms).

In [ ]:
rng = np.random.default_rng(1998)
isi_exp = rng.exponential(0.05, 200000)
isi_gam = rng.gamma(4.0, 0.05 / 4, 200000)

def survival_pair(isi):
    uncond = ...                      # P(wait > 0.05)
    cond = ...                        # P(wait > 0.10 | wait > 0.05)
    return uncond, cond

uncond_exp, cond_exp = survival_pair(isi_exp)
uncond_gam, cond_gam = survival_pair(isi_gam)
print(f"exponential: P(>50 ms) = {uncond_exp:.3f}, P(>100 | >50) = {cond_exp:.3f}")
print(f"gamma(4):    P(>50 ms) = {uncond_gam:.3f}, P(>100 | >50) = {cond_gam:.3f}")
print("coefficients of variation:", round(isi_exp.std() / isi_exp.mean(), 3), round(isi_gam.std() / isi_gam.mean(), 3))

In [ ]:
import numpy as np
def test_exponential():
    "for exponential intervals, P(wait > 100 | wait > 50) matches P(wait > 50)"
    assert abs(cond_exp - uncond_exp) < 0.03

def test_gamma():
    "for gamma intervals of the same mean, having waited 50 ms makes a long further wait less likely"
    assert cond_gam < uncond_gam - 0.1

test_exponential()
test_gamma()
print("All checks passed")

### Exercise: Pooled counts

Fifty independent neurons with rates drawn between 1 and 10 spikes per 100 ms window each give 5,000 Poisson counts. Sum across neurons to get `pooled`, and compare its mean and Fano factor with the total rate `lam_total`.

In [ ]:
rates = rng.uniform(1, 10, 50)
counts50 = rng.poisson(rates, (5000, 50))
pooled = ...
lam_total = ...
print(f"pooled mean {pooled.mean():.2f} (total rate {lam_total:.2f}), Fano factor {pooled.var() / pooled.mean():.3f}")

In [ ]:
import numpy as np
def test_pooled():
    "the pooled count has mean close to the summed rate and a Fano factor close to 1"
    assert abs(pooled.mean() - lam_total) / lam_total < 0.02 and abs(pooled.var() / pooled.mean() - 1) < 0.05

test_pooled()
print("All checks passed")

### Exercise: Partition the variability

A neuron is tested at 8 orientations, 300 trials each, with a gamma-distributed gain of variance 0.1 on every trial. Compute the mean `mu` and variance `var` at each orientation, the Fano factors `fano`, and fit σ² in Var = μ + σ²μ² by least squares on (Var − μ) against μ².

In [ ]:
thetas = np.linspace(0, np.pi, 8, endpoint=False)
f = 2 + 18 * np.exp(2 * (np.cos(2 * (thetas - np.pi / 2)) - 1))    # tuning curve: mean count per trial
sigma2 = 0.1
gain = rng.gamma(1 / sigma2, sigma2, (300, 8))                      # mean 1, variance 0.1
K = rng.poisson(gain * f)

mu = ...
var = ...
fano = var / mu
sigma2_hat = ...                                                    # least squares slope through the origin
print("mean:", mu.round(1)); print("Fano:", fano.round(2)); print(f"fitted gain variance {sigma2_hat:.3f} (true 0.1)")
m = np.linspace(0, mu.max() * 1.1, 100)
plt.plot(mu, var, "o"); plt.plot(m, m, "k--", label="Poisson"); plt.plot(m, m + sigma2_hat * m ** 2, label="modulated Poisson")
plt.xlabel("mean count"); plt.ylabel("variance"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_super_poisson():
    "the Fano factor rises with the mean, and is above 1 at the preferred orientation"
    assert fano[np.argmax(mu)] > 1.3 and np.corrcoef(mu, fano)[0, 1] > 0.9

def test_fit():
    "the fitted gain variance is within 25% of the true 0.1"
    assert abs(sigma2_hat - 0.1) / 0.1 < 0.25

test_super_poisson()
test_fit()
print("All checks passed")